# Praktikum Prompt Engineering — Student Graduation Prediction (Kelompok B)

Notebook perbaikan untuk menjalankan analisis dan model tanpa memasang ulang `pandas`/`numpy`.
**Penting:** jalankan berurutan dari atas ke bawah. Jangan jalankan `pip install -r requirements.txt` di notebook ini karena dapat mengganti versi paket bawaan Google Colab.

## 1. Persiapan environment

Cell ini hanya memeriksa versi library. Tidak ada instalasi atau downgrade paket.

In [1]:
import sys
import pandas as pd
import numpy as np
import sklearn

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Environment siap.")

Python: 3.13.18
pandas: 2.2.3
numpy: 2.4.6
scikit-learn: 1.6.1
Environment siap.


## 2. Ambil file proyek

Jika folder proyek sudah ada di `/content/project/Kelompok-B-Vibe-Coding`, cell akan langsung menggunakannya.
Jika belum ada, cell meminta ZIP proyek untuk diunggah.

In [2]:
from pathlib import Path
import zipfile

PROJECT_DIR = Path("/content/project/Kelompok-B-Vibe-Coding")
ZIP_PATH = Path("/content/Kelompok-B-Vibe-Coding.zip")

if not PROJECT_DIR.exists():
    from google.colab import files
    print("Folder proyek belum ditemukan. Silakan upload ZIP proyek Kelompok B.")
    uploaded = files.upload()
    zip_candidates = [Path("/content") / name for name in uploaded.keys() if name.lower().endswith(".zip")]
    if not zip_candidates:
        raise FileNotFoundError("File ZIP tidak ditemukan. Upload file proyek dalam format .zip.")
    ZIP_PATH = zip_candidates[0]
    extract_root = Path("/content/project")
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(extract_root)
    candidates = list(extract_root.rglob("siswa.csv"))
    if not candidates:
        raise FileNotFoundError("ZIP berhasil diekstrak, tetapi data/siswa.csv tidak ditemukan.")
    PROJECT_DIR = candidates[0].parent.parent

DATA_PATH = PROJECT_DIR / "data" / "siswa.csv"
if not DATA_PATH.exists():
    candidates = list(PROJECT_DIR.rglob("siswa.csv"))
    if not candidates:
        raise FileNotFoundError(f"Dataset siswa.csv tidak ditemukan di {PROJECT_DIR}")
    DATA_PATH = candidates[0]

print("Folder proyek:", PROJECT_DIR)
print("Dataset:", DATA_PATH)
print("Dataset tersedia:", DATA_PATH.exists())

Folder proyek belum ditemukan. Silakan upload ZIP proyek Kelompok B.


Saving Kelompok-B-Vibe-Coding.zip to Kelompok-B-Vibe-Coding (1).zip
Folder proyek: /content/project/Kelompok-B-Vibe-Coding
Dataset: /content/project/Kelompok-B-Vibe-Coding/data/siswa.csv
Dataset tersedia: True


## 3. Membaca dataset

Gunakan `to_string()` agar Colab tidak mencoba merender DataFrame dengan formatter HTML yang sedang bermasalah.

In [3]:
df = pd.read_csv(DATA_PATH)
print("Ukuran dataset:", df.shape)
print("Kolom:", df.columns.tolist())
print("\n5 baris pertama:")
print(df.head().to_string(index=False))

Ukuran dataset: (615, 8)
Kolom: ['id_siswa', 'program', 'jam_belajar', 'kehadiran', 'nilai_uts', 'nilai_tugas', 'lulus', 'nilai_remedial']

5 baris pertama:
id_siswa program  jam_belajar  kehadiran  nilai_uts  nilai_tugas  lulus  nilai_remedial
   S0000    ipa           6.1       88.0       72.0         68.0      1             NaN
   S0001  BAHASA          6.2       93.0       82.0         60.0      1             NaN
   S0002     IPS         12.9       77.0       79.0         76.0      1             NaN
   S0003     IPS          6.3       94.0       69.0         57.0      0            56.0
   S0004     IPS          8.0       71.0       64.0         73.0      1             NaN


## 4. Audit data

In [4]:
print("Informasi dataset:")
print(df.info())

print("\nMissing value per kolom:")
print(df.isna().sum().to_string())

print("\nJumlah baris duplikat:", int(df.duplicated().sum()))
if "id_siswa" in df.columns:
    print("Jumlah ID duplikat:", int(df["id_siswa"].duplicated().sum()))

print("\nDistribusi target lulus:")
print(df["lulus"].value_counts(dropna=False).to_string())
print("\nDistribusi target (%):")
print((df["lulus"].value_counts(normalize=True, dropna=False) * 100).round(2).to_string())

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 615 entries, 0 to 614
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_siswa        615 non-null    object 
 1   program         615 non-null    object 
 2   jam_belajar     579 non-null    float64
 3   kehadiran       615 non-null    float64
 4   nilai_uts       591 non-null    float64
 5   nilai_tugas     615 non-null    float64
 6   lulus           615 non-null    int64  
 7   nilai_remedial  251 non-null    float64
dtypes: float64(5), int64(1), object(2)
memory usage: 38.6+ KB
None

Missing value per kolom:
id_siswa            0
program             0
jam_belajar        36
kehadiran           0
nilai_uts          24
nilai_tugas         0
lulus               0
nilai_remedial    364

Jumlah baris duplikat: 15
Jumlah ID duplikat: 15

Distribusi target lulus:
lulus
1    364
0    251

Distribusi target (%):
lulus
1    59.19
0    40.81


## 5. Pembersihan dan EDA
`nilai_remedial` tidak digunakan sebagai fitur karena pada data ini nilainya berkaitan langsung dengan status kelulusan (berpotensi menyebabkan target leakage). `id_siswa` juga tidak digunakan sebagai fitur karena hanya berfungsi sebagai identitas.

In [5]:
df_clean = df.drop_duplicates().copy()

if "program" in df_clean.columns:
    df_clean["program"] = df_clean["program"].astype("string").str.strip().str.upper()

print("Ukuran setelah menghapus duplikat:", df_clean.shape)
print("\nDistribusi program:")
print(df_clean["program"].value_counts(dropna=False).to_string())

print("\nRingkasan statistik numerik:")
print(df_clean.describe(include="number").round(2).to_string())

Ukuran setelah menghapus duplikat: (600, 8)

Distribusi program:
program
IPA       207
BAHASA    203
IPS       190

Ringkasan statistik numerik:
       jam_belajar  kehadiran  nilai_uts  nilai_tugas   lulus  nilai_remedial
count       564.00     600.00     576.00       600.00  600.00          249.00
mean          7.40      83.27      70.47        75.27    0.58           59.87
std           3.23       9.53      12.00         9.91    0.49            8.16
min           1.10      55.00      33.00        46.00    0.00           39.00
25%           5.10      77.00      62.75        68.00    0.00           54.00
50%           7.00      84.00      71.00        75.00    1.00           60.00
75%           9.30      90.00      78.25        82.00    1.00           65.00
max          23.20     100.00     100.00       100.00    1.00           88.00


## 6. Menyiapkan fitur dan pipeline

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

target = "lulus"
exclude_cols = [target, "id_siswa", "nilai_remedial"]
X = df_clean.drop(columns=[c for c in exclude_cols if c in df_clean.columns])
y = df_clean[target].astype(int)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000))
])

print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)
print("Data train:", X_train.shape)
print("Data test:", X_test.shape)

Fitur numerik: ['jam_belajar', 'kehadiran', 'nilai_uts', 'nilai_tugas']
Fitur kategorikal: ['program']
Data train: (480, 5)
Data test: (120, 5)


## 7. Training dan evaluasi model

In [7]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.675

Classification report:
              precision    recall  f1-score   support

           0       0.68      0.42      0.52        50
           1       0.67      0.86      0.75        70

    accuracy                           0.68       120
   macro avg       0.68      0.64      0.64       120
weighted avg       0.68      0.68      0.66       120

Confusion matrix:
[[21 29]
 [10 60]]


## 8. Uji satu prediksi
Contoh ini mengambil satu baris dari data uji. Ini hanya demonstrasi prediksi, bukan klaim bahwa model sudah layak digunakan di dunia nyata.

In [8]:
sample = X_test.iloc[[0]]
prediction = int(model.predict(sample)[0])
probability = float(model.predict_proba(sample)[0][1])

print("Input contoh:")
print(sample.to_string(index=False))
print("Prediksi:", "Lulus" if prediction == 1 else "Tidak Lulus")
print("Probabilitas kelas Lulus:", f"{probability:.2%}")

Input contoh:
program  jam_belajar  kehadiran  nilai_uts  nilai_tugas
    IPS          5.3       89.0       69.0         72.0
Prediksi: Lulus
Probabilitas kelas Lulus: 55.53%


## 9. Catatan hasil dan keterbatasan

- Model yang digunakan adalah Logistic Regression sebagai baseline.
- Evaluasi harus dibaca dari accuracy, precision, recall, F1-score, dan confusion matrix; jangan hanya melihat accuracy.
- Perhatikan recall kelas `0` (Tidak Lulus), karena kesalahan memprediksi siswa tidak lulus sebagai lulus dapat berdampak penting.
- Hasil model bergantung pada kualitas dan keterbatasan dataset; hasil ini bukan keputusan kelulusan final.

## 10. Empat contoh kesalahan kode AI dan perbaikannya

| Kesalahan yang perlu diaudit | Risiko | Perbaikan |
|---|---|---|
| Mengisi semua missing value dengan `0` | Mengubah makna data dan dapat menimbulkan bias | Imputasi median untuk numerik dan modus untuk kategorikal |
| Melakukan scaling sebelum train-test split | Data leakage dari data test | Letakkan imputasi dan scaling di dalam `Pipeline` |
| Membagi data tanpa `stratify` dan `random_state` | Proporsi kelas berubah dan hasil tidak reproducible | Gunakan `stratify=y` dan `random_state=42` |
| Memasukkan `id_siswa` atau `nilai_remedial` sebagai fitur | ID tidak informatif; remedial berpotensi membocorkan target | Keluarkan keduanya dari fitur |

## 11. Prompt log (contoh rekonstruksi)

Tandai sebagai rekonstruksi jika prompt tidak disimpan persis saat proses awal.

In [9]:
prompt_log = [
    [1, "B", "[Rekonstruksi] Implementasikan preprocessing berdasarkan SPEC.md dan EDA_REPORT.md",
     "Modul preprocessing berhasil dibuat", "Missing value, duplikasi, potensi data leakage",
     "Menggunakan imputasi sesuai tipe data dan pipeline"],
    [2, "B", "[Rekonstruksi] Terapkan train-test split dengan stratify dan random_state 42",
     "Data training dan testing berhasil dipisahkan", "Proporsi kelas perlu dijaga",
     "Menggunakan stratify dan random_state=42"],
    [3, "B", "[Rekonstruksi] Bangun Logistic Regression dan lakukan evaluasi",
     "Model baseline berhasil dilatih dan dievaluasi", "Recall kelas tidak lulus perlu diperhatikan",
     "Melaporkan metrik kedua kelas dan confusion matrix"],
    [4, "B", "[Rekonstruksi] Buat aplikasi prediksi kelulusan menggunakan Gradio",
     "Aplikasi Gradio dirancang", "Input pengguna perlu divalidasi",
     "Menambahkan validasi input dan prediksi probabilitas"],
    [5, "B", "[Rekonstruksi] Audit kode dan perbaiki temuan yang relevan",
     "Perbaikan kode berhasil diterapkan", "Potensi masalah validasi dan error handling",
     "Memperbaiki kode tanpa mengubah dataset dan algoritma"]
]
print("Jumlah iterasi prompt:", len(prompt_log))
for row in prompt_log:
    print(" | ".join(map(str, row)))

Jumlah iterasi prompt: 5
1 | B | [Rekonstruksi] Implementasikan preprocessing berdasarkan SPEC.md dan EDA_REPORT.md | Modul preprocessing berhasil dibuat | Missing value, duplikasi, potensi data leakage | Menggunakan imputasi sesuai tipe data dan pipeline
2 | B | [Rekonstruksi] Terapkan train-test split dengan stratify dan random_state 42 | Data training dan testing berhasil dipisahkan | Proporsi kelas perlu dijaga | Menggunakan stratify dan random_state=42
3 | B | [Rekonstruksi] Bangun Logistic Regression dan lakukan evaluasi | Model baseline berhasil dilatih dan dievaluasi | Recall kelas tidak lulus perlu diperhatikan | Melaporkan metrik kedua kelas dan confusion matrix
4 | B | [Rekonstruksi] Buat aplikasi prediksi kelulusan menggunakan Gradio | Aplikasi Gradio dirancang | Input pengguna perlu divalidasi | Menambahkan validasi input dan prediksi probabilitas
5 | B | [Rekonstruksi] Audit kode dan perbaiki temuan yang relevan | Perbaikan kode berhasil diterapkan | Potensi masalah valid

## 12. Checklist dan refleksi

Centang checklist setelah benar-benar memeriksa hasilnya.

In [12]:
checklist = {
    "Dataset berhasil dibaca dan diaudit": True,
    "Preprocessing dilakukan dalam Pipeline setelah train-test split": True,
    "Model berhasil dilatih dan dievaluasi": True,
    "Empat contoh kesalahan kode AI dan perbaikannya didokumentasikan": True,
    "Minimal lima iterasi prompt didokumentasikan (rekonstruksi diberi label)": True,
    "Notebook dapat dijalankan ulang dari awal dengan random_state tetap": True,
    "Saya memahami gambaran kompleksitas waktu proses": True
}
for item, done in checklist.items():
    print(("[x]" if done else "[ ]"), item)

[x] Dataset berhasil dibaca dan diaudit
[x] Preprocessing dilakukan dalam Pipeline setelah train-test split
[x] Model berhasil dilatih dan dievaluasi
[x] Empat contoh kesalahan kode AI dan perbaikannya didokumentasikan
[x] Minimal lima iterasi prompt didokumentasikan (rekonstruksi diberi label)
[x] Notebook dapat dijalankan ulang dari awal dengan random_state tetap
[x] Saya memahami gambaran kompleksitas waktu proses


**Refleksi (silakan sesuaikan dengan pengalaman sebenarnya):**

Selama pengembangan proyek, saya menemukan bahwa kode AI dapat melakukan preprocessing sebelum pembagian data training dan testing sehingga berpotensi menyebabkan data leakage. Hal tersebut diketahui melalui pemeriksaan alur preprocessing dan evaluasi kode. Untuk mengatasinya, saya menggunakan Pipeline dan ColumnTransformer agar proses imputasi dan scaling dipelajari dari data training. Saya juga memahami bahwa hasil evaluasi perlu dilihat dari beberapa metrik, terutama recall untuk kelas Tidak Lulus, bukan hanya accuracy.

## 13. Simpan hasil model (opsional)

In [11]:
import joblib
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_path = OUTPUT_DIR / "graduation_model_notebook.joblib"
joblib.dump(model, model_path)
print("Model tersimpan di:", model_path)

Model tersimpan di: /content/project/Kelompok-B-Vibe-Coding/outputs/graduation_model_notebook.joblib
